# Week 7 Lab: When Elimination Goes Wrong
**MATH-3515 Numerical Analysis** &nbsp;·&nbsp; Estimated time: **60 minutes**

In this lab you will build a linear-system solver, break it, and then find a problem that nothing seems to fix.

## Learning outcomes
By the end of this lab, you will be able to:

1. **Implement** Gaussian elimination with optional partial pivoting and back substitution in Python, translating the Week 7 pseudocode into a working solver.
2. **Verify** a numerical solver by testing it on a problem with a known answer and comparing its output with a library routine (`np.linalg.solve`).
3. **Explain** how a tiny pivot combined with floating-point rounding destroys accuracy in elimination without pivoting, tracing the loss of information to a specific step of the algorithm.
4. **Compute and distinguish** the absolute error, $\max_k |x_k-\hat x_k|$, and the absolute residual, $\max_k |b_k-(A\hat{\mathbf x})_k|$, of a computed solution.
5. **Analyze** tables and plots of error against problem size to describe how error grows, and **recognize** that a solution can have a residual near machine precision and still be inaccurate.
6. **Compare** the effect of partial pivoting on two different systems, and **hypothesize** why pivoting fixes one failure but not the other.

## How to work through this notebook
- Run the cells from top to bottom (**Shift + Enter**).
- Replace every `____` (four underscores) with your own code.
- Cells marked **Check your work** test your answers. If a check fails, fix the cell above it before moving on.
- Cells marked **Your answer** are for a short written response (one to three sentences).

| Part | Topic | Time |
|---|---|---|
| 0 | Setup | 2 min |
| 1 | Your solver | 15 min |
| 2 | The tiny-pivot example | 14 min |
| 3 | The Hilbert experiment | 19 min |
| 4 | Small residual, big error | 10 min |

## Part 0. Setup (2 min)
Run this cell. It loads the libraries and prints machine epsilon $\varepsilon$ from Week 2.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=False)
eps = np.finfo(float).eps
print("machine epsilon =", eps)

machine epsilon = 2.220446049250313e-16


## Part 1. Your solver (15 min)

Below is the Week 6 algorithm, **Gaussian elimination with back substitution**, with partial pivoting as an option.

**Forward elimination.** For $k = 1, \dots, n-1$:
- *(pivoting only)* find the row $r \ge k$ with the largest $|A_{rk}|$ and swap rows $k$ and $r$ of $A$ and entries $k$ and $r$ of $\mathbf b$;
- if the pivot $A_{kk}$ is $0$, stop: the matrix is singular;
- for each row $i = k+1, \dots, n$: compute the multiplier $m = A_{ik}/A_{kk}$, then update
$$A_{ij} \leftarrow A_{ij} - m\,A_{kj}\ \ (j = k+1,\dots,n), \qquad b_i \leftarrow b_i - m\,b_k .$$

**Back substitution.** For $i = n, n-1, \dots, 1$:
$$x_i = \frac{b_i - \sum_{j=i+1}^{n} A_{ij}\,x_j}{A_{ii}} .$$

Python counts from 0, so row 1 of the algorithm is row `0` in the code.

**Task 1.1.** Fill in the four blanks.

In [2]:
def gauss_solve(A, b, pivot=True):
    '''Solve A x = b by Gaussian elimination and back substitution.
    pivot=True uses partial pivoting; pivot=False never swaps rows.'''
    A = np.array(A, dtype=float)      # work on copies so the inputs are not changed
    b = np.array(b, dtype=float)
    n = len(b)

    # ---- forward elimination ----
    for k in range(n - 1):
        if pivot:
            r = k + np.argmax(np.abs(A[k:, k]))   # row with the largest |A[r, k]|
            A[[k, r]] = A[[r, k]]                 # swap rows k and r of A
            b[[k, r]] = b[[r, k]]                 # swap entries k and r of b
        if A[k, k] == 0:
            raise ValueError("zero pivot: matrix is singular or needs a row swap")
        for i in range(k + 1, n):
            m = A[i, k] / A[k, k]
            A[i, k] = 0.0
            for j in range(k + 1, n):
                A[i, j] = ____                    # blank 1: update entry (i, j)
            b[i] = ____                           # blank 2: update entry i of b
    if A[n - 1, n - 1] == 0:
        raise ValueError("zero pivot: matrix is singular")

    # ---- back substitution ----
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        s = b[i]
        for j in range(i + 1, n):
            s = ____                              # blank 3: subtract the known terms
        x[i] = ____                               # blank 4: divide by the pivot
    return x

**Task 1.2. Test on a problem with a known answer.** In Week 6 (Example 4) we solved
$$\begin{pmatrix}1&-2\\5&-14\end{pmatrix}\mathbf x=\begin{pmatrix}1\\1\end{pmatrix}
\quad\text{and found}\quad \mathbf x=\begin{pmatrix}3\\1\end{pmatrix}.$$
With pivoting turned on, your solver swaps the two rows first (because $|5|>|1|$). The answer must not change.

In [3]:
A1 = np.array([[1, -2], [5, -14]])
b1 = np.array([1, 1])
print("with pivoting:   ", gauss_solve(A1, b1, pivot=True))
print("without pivoting:", gauss_solve(A1, b1, pivot=False))

NameError: name '____' is not defined

In [4]:
# Check your work
assert np.allclose(gauss_solve(A1, b1, pivot=True),  [3, 1]), "pivot=True should give (3, 1)"
assert np.allclose(gauss_solve(A1, b1, pivot=False), [3, 1]), "pivot=False should give (3, 1)"
print("Task 1.2 passed.")

NameError: name '____' is not defined

**Task 1.3. A zero pivot.** The system
$$\begin{pmatrix}0&1\\1&0\end{pmatrix}\mathbf x=\begin{pmatrix}2\\3\end{pmatrix}$$
has the solution $\mathbf x=(3,2)$, but its first pivot is $0$. Run the cell and read both lines of output.

In [5]:
A2 = np.array([[0, 1], [1, 0]])
b2 = np.array([2, 3])

try:
    print("without pivoting:", gauss_solve(A2, b2, pivot=False))
except ValueError as err:
    print("without pivoting: FAILED ->", err)

print("with pivoting:   ", gauss_solve(A2, b2, pivot=True))

without pivoting: FAILED -> zero pivot: matrix is singular or needs a row swap


NameError: name '____' is not defined

**Task 1.4. Compare with a library routine.** `np.linalg.solve` is NumPy's built-in solver. Fill in the blank to solve the same random system with your function, then compare.

In [6]:
rng = np.random.default_rng(3515)
A3 = rng.standard_normal((6, 6))
b3 = rng.standard_normal(6)

x_mine = ____                      # blank 5: solve A3 x = b3 with your function
x_numpy = np.linalg.solve(A3, b3)
print("largest difference from NumPy:", np.max(np.abs(x_mine - x_numpy)))

NameError: name '____' is not defined

In [ ]:
# Check your work
assert np.allclose(x_mine, x_numpy), "your solver disagrees with np.linalg.solve"
print("Task 1.4 passed. Your solver is ready.")

## Part 2. The tiny-pivot example (14 min)

Consider, for a small number $\delta > 0$,
$$\begin{pmatrix}\delta&1\\1&1\end{pmatrix}\mathbf x=\begin{pmatrix}1\\2\end{pmatrix},
\qquad\text{exact solution}\qquad
x_1=\frac{1}{1-\delta},\quad x_2=\frac{1-2\delta}{1-\delta}.$$
For tiny $\delta$ the exact solution is very close to $(1,1)$.

**Task 2.1.** Run the cell for $\delta=10^{-20}$.

In [ ]:
delta = 1e-20
A = np.array([[delta, 1.0], [1.0, 1.0]])
b = np.array([1.0, 2.0])

print("without pivoting:", gauss_solve(A, b, pivot=False))
print("with pivoting:   ", gauss_solve(A, b, pivot=True))

**Task 2.2. Trace the failure by hand.** Without pivoting, elimination uses the multiplier $m = 1/\delta = 10^{20}$ and updates
$$A_{22} \leftarrow 1 - 10^{20}\cdot 1, \qquad b_2 \leftarrow 2 - 10^{20}\cdot 1 .$$
What does the computer store for $1-10^{20}$? (Hint: compare $1$ with $\varepsilon\cdot 10^{20}$.) Where does the information about the original entries go, and how does that make $x_1$ wrong?

Test your idea in the next cell.

In [ ]:
print("1 - 1e20      =", 1 - 1e20)
print("(1 - 1e20) == -1e20 ?", (1 - 1e20) == -1e20)
print("eps * 1e20    =", eps * 1e20)

**Your answer (Task 2.2):**

**Task 2.3. How small must $\delta$ be?** The loop below solves the system for $\delta = 10^{-2}, 10^{-4}, \dots, 10^{-20}$, both ways. Fill in the one blank: the **absolute error** of a computed solution `xhat` compared with the exact solution `x_exact`, measured as the largest entry of $|\mathbf x-\hat{\mathbf x}|$.

In [ ]:
def abs_error(x_exact, xhat):
    return ____                         # blank 6: largest entry of |x_exact - xhat|

deltas = 10.0 ** -np.arange(2, 21, 2)
err_nopiv, err_piv = [], []

print(f"{'delta':>8} | {'error, no pivoting':>18} | {'error, pivoting':>15}")
print("-" * 48)
for d in deltas:
    A = np.array([[d, 1.0], [1.0, 1.0]])
    b = np.array([1.0, 2.0])
    x_exact = np.array([1 / (1 - d), (1 - 2 * d) / (1 - d)])
    e0 = abs_error(x_exact, gauss_solve(A, b, pivot=False))
    e1 = abs_error(x_exact, gauss_solve(A, b, pivot=True))
    err_nopiv.append(e0)
    err_piv.append(e1)
    print(f"{d:8.0e} | {e0:18.1e} | {e1:15.1e}")

In [ ]:
# Check your work
assert abs_error(np.array([1.0, 2.0]), np.array([1.5, 1.0])) == 1.0, "abs_error should return the largest |difference|"
assert abs_error(np.array([1.0, 2.0]), np.array([3.0, 2.0])) == 2.0, "use the absolute value of the difference"
print("Task 2.3 passed.")

**Task 2.4.** Run the plot. The dashed line is $\varepsilon/\delta$. (Errors that are exactly $0$ cannot be drawn on a log scale, so they are shown at $10^{-17}$.)

In [ ]:
floor = 1e-17
plt.figure(figsize=(6.5, 4))
plt.loglog(deltas, np.maximum(err_nopiv, floor), "o-", label="no pivoting")
plt.loglog(deltas, np.maximum(err_piv, floor), "s-", label="partial pivoting")
plt.loglog(deltas, eps / deltas, "k--", lw=1, label=r"$\varepsilon/\delta$")
plt.gca().invert_xaxis()
plt.xlabel(r"$\delta$  (smaller to the right)")
plt.ylabel("absolute error")
plt.title("Tiny pivot: error with and without pivoting")
plt.legend()
plt.grid(True, which="major", alpha=0.3)
plt.show()

**Your answer (Task 2.4):** Describe how the error without pivoting grows as $\delta$ shrinks. Was the failure caused by the matrix or by the algorithm? Explain.

## Part 3. The Hilbert experiment (19 min)

The **Hilbert matrix** $H_n$ is the $n\times n$ matrix with entries
$$(H_n)_{ij}=\frac{1}{i+j-1}, \qquad i,j=1,\dots,n .$$
For example,
$$H_3=\begin{pmatrix}1&\tfrac12&\tfrac13\\ \tfrac12&\tfrac13&\tfrac14\\ \tfrac13&\tfrac14&\tfrac15\end{pmatrix}.$$

**Task 3.1.** Fill in the blank. Remember that Python indices `i, j` start at 0, so the math indices are `i+1` and `j+1`.

In [ ]:
def hilbert(n):
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            H[i, j] = ____              # blank 7: the (i, j) entry
    return H

print(hilbert(3))

In [ ]:
# Check your work
from scipy.linalg import hilbert as scipy_hilbert
assert np.allclose(hilbert(3), [[1, 1/2, 1/3], [1/2, 1/3, 1/4], [1/3, 1/4, 1/5]]), "check the formula for H[i, j]"
assert np.allclose(hilbert(10), scipy_hilbert(10)), "hilbert(10) does not match SciPy"
print("Task 3.1 passed.")

**Task 3.2. The experiment.** For each $n$:
1. choose the exact answer $\mathbf x=(1,1,\dots,1)$;
2. compute $\mathbf b=H_n\mathbf x$;
3. solve $H_n\hat{\mathbf x}=\mathbf b$ with **your** solver, **with pivoting**;
4. record the **absolute error** $\max_k|x_k-\hat x_k|$ and the **absolute residual** $\max_k|b_k-(H_n\hat{\mathbf x})_k|$.

Fill in the three blanks.

In [ ]:
ns = [2, 4, 6, 8, 10, 12, 14]
hil_err, hil_res = [], []

print(f"{'n':>3} | {'absolute error':>15} | {'absolute residual':>18}")
print("-" * 43)
for n in ns:
    H = hilbert(n)
    x = np.ones(n)
    b = ____                             # blank 8: right-hand side b = H x
    xhat = gauss_solve(H, b, pivot=True)
    err = ____                           # blank 9: absolute error
    res = ____                           # blank 10: absolute residual
    hil_err.append(err)
    hil_res.append(res)
    print(f"{n:3d} | {err:15.1e} | {res:18.1e}")

In [ ]:
# Check your work
assert len(hil_err) == len(ns) and len(hil_res) == len(ns), "run the experiment cell first"
assert hil_err[0] < 1e-12 and hil_err[-1] > 1e-2, "the error should start tiny and end large"
assert max(hil_res) < 1e-12, "the residual should stay tiny"
print("Task 3.2 passed.")

**Task 3.3.** Run the plot of your table. (A residual that is exactly $0$ is shown at $10^{-17}$.)

In [ ]:
plt.figure(figsize=(6.5, 4))
plt.semilogy(ns, np.maximum(hil_err, 1e-17), "o-", label="absolute error")
plt.semilogy(ns, np.maximum(hil_res, 1e-17), "s-", label="absolute residual")
plt.axhline(eps, color="k", ls="--", lw=1, label=r"$\varepsilon$")
plt.xlabel("n")
plt.ylabel("size (log scale)")
plt.title("Hilbert systems solved with partial pivoting")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Task 3.4. Does pivoting help here?** The Hilbert matrix never has a tiny pivot. Run the cell to repeat the experiment with and without pivoting.

In [ ]:
print(f"{'n':>3} | {'error, pivoting':>15} | {'error, no pivoting':>18}")
print("-" * 43)
for n in ns:
    H = hilbert(n)
    x = np.ones(n)
    b = H @ x
    e_piv = np.max(np.abs(x - gauss_solve(H, b, pivot=True)))
    e_nopiv = np.max(np.abs(x - gauss_solve(H, b, pivot=False)))
    print(f"{n:3d} | {e_piv:15.1e} | {e_nopiv:18.1e}")

**Your answer (Task 3.4):** The computed $\hat{\mathbf x}$ satisfies the equations almost perfectly (tiny residual). Is it a good answer? Did pivoting make a difference?

## Part 4. Small residual, big error (10 min)

A $2\times2$ system lets us **see** how this can happen. Consider
$$\begin{pmatrix}1&1\\1&1.0001\end{pmatrix}\mathbf x=\begin{pmatrix}2\\2.0001\end{pmatrix},
\qquad\text{exact solution } \mathbf x=(1,1).$$

**Task 4.1.** Someone proposes the guess $\hat{\mathbf x}=(2,0)$. Fill in the blanks to compute the residual vector $\mathbf r=\mathbf b-A\hat{\mathbf x}$ and the error vector $\mathbf e=\mathbf x-\hat{\mathbf x}$.

In [ ]:
A = np.array([[1.0, 1.0], [1.0, 1.0001]])
b = np.array([2.0, 2.0001])
x_exact = np.array([1.0, 1.0])
x_guess = np.array([2.0, 0.0])

r = ____                                 # blank 11: residual vector
e = ____                                 # blank 12: error vector
print("residual r =", r, "   absolute residual =", np.max(np.abs(r)))
print("error    e =", e, "   absolute error    =", np.max(np.abs(e)))

In [ ]:
# Check your work
assert np.allclose(r, [0, -1e-4]) or np.allclose(r, [0, 1e-4]), "check the residual formula r = b - A x_guess"
assert np.allclose(e, [-1, 1]), "check the error formula e = x_exact - x_guess"
print("Task 4.1 passed.")

**Task 4.2.** Each equation is a line in the $(x_1,x_2)$ plane. Run the cell to draw both lines, the exact solution, and the guess.

In [ ]:
x1 = np.linspace(-0.5, 3, 200)
line1 = 2 - x1                          # x1 + x2 = 2
line2 = (2.0001 - x1) / 1.0001          # x1 + 1.0001 x2 = 2.0001

plt.figure(figsize=(6.5, 4))
plt.plot(x1, line1, lw=3, alpha=0.6, label=r"$x_1 + x_2 = 2$")
plt.plot(x1, line2, "--", lw=1.5, label=r"$x_1 + 1.0001\,x_2 = 2.0001$")
plt.plot(*x_exact, "o", ms=9, label="exact solution (1, 1)")
plt.plot(*x_guess, "X", ms=10, label="guess (2, 0)")
plt.xlabel(r"$x_1$")
plt.ylabel(r"$x_2$")
plt.title("Two nearly parallel lines")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Your answer (Task 4.2):** Use the picture to explain how the guess can have a tiny residual and a large error.

## Summary

| Experiment | Residual | Error | Did pivoting fix it? |
|---|---|---|---|
| Part 2: tiny pivot $\delta$ | — | large without pivoting | **Yes** |
| Part 3: Hilbert $H_n$ | $\approx 10^{-16}$ | grows to order 1 and beyond | **No** |
| Part 4: nearly parallel lines | $10^{-4}$ | $1$ | — |

Two different things went wrong today. In Part 2 the **algorithm** failed, and a better algorithm fixed it. In Part 3 the algorithm did its job (the residual is at machine precision), and the answer was still wrong. In Week 8 we will find a single number that predicts this behavior.

## Optional extension: the cost of elimination (not graded)

In Week 7 we counted $T(n)\approx\tfrac23 n^3$ operations for elimination. If that is right, doubling $n$ should multiply the running time by about $2^3=8$. Run the cell to test the prediction with your solver.

In [ ]:
import time

sizes = [40, 80, 160]
times = []
rng = np.random.default_rng(0)
for n in sizes:
    A = rng.standard_normal((n, n))
    b = rng.standard_normal(n)
    t0 = time.perf_counter()
    gauss_solve(A, b)
    times.append(time.perf_counter() - t0)

for k, n in enumerate(sizes):
    ratio = "" if k == 0 else f"   ratio to previous: {times[k] / times[k-1]:.1f}"
    print(f"n = {n:4d}: {times[k]:.3f} s{ratio}")